# Molecular Combinatorial Shift Keying (MoCSK) Framework

Unified, modular pipeline for molecular communication simulation, modulation, decoding, and benchmarking.

### Supported Modulation Schemes:
1. **Permutation MoCSK**: Molecules released in order of a permutation; decoded by smoothed peak arrival order.
2. **Combination MoCSK (Adaptive)**: Subset of molecule alphabet; decoded via dynamic gap-based threshold.
3. **Combination MoCSK (Fixed)**: Pre-calibrated global threshold from label distributions.
4. **Sparse Pattern MoCSK**: Sparse slot allocation with molecule exclusion to eliminate ISI tails.

All schemes use the unified physical channel and signal processing utilities.

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

# Ensure repository root is on sys.path
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import config
from modulation import (
    run_permutation,
    run_combination_adaptive,
    run_combination_fixed,
    run_sparse,
    get_all_permutations,
    get_all_combinations,
    get_all_sparse_patterns,
)
from channel.diffusion import ensure_experiment_templates
from experiments.sweep import run_ser_vs_ts_sweep, run_ber_vs_ts_sweep

print("Physical Configuration:")
print(f"  Distance r0           : {config.R0} µm")
print(f"  Receiver radius rr    : {config.RR} µm")
print(f"  Diffusion coef D      : {config.D} µm²/s")
print(f"  Theoretical Peak Time : {config.T_PEAK_THEORY:.4f} s")
print(f"  Template directory    : {config.CUSTOM_DIR}")

## 1. Permutation Scheme
Each '1' bit carries a random full permutation of all $K$ molecule types. Decoded by finding the arrival peak order.

In [ ]:
test_bits = "10110"

res_perm = run_permutation(
    bit_sequence=test_bits,
    num_molecule_types=5,
    delay_between_symbols=2.5,
    delay_between_molecules=0.4,
    plot=True,
)

## 2. Combination Scheme (Adaptive Threshold)
Each '1' bit carries a subset of molecule types. Decoded by splitting the largest gap in smoothed peaks.

In [ ]:
res_combo_adapt = run_combination_adaptive(
    bit_sequence=test_bits,
    num_molecule_types=5,
    delay_between_symbols=2.5,
    delay_between_molecules=0.4,
    plot=True,
)

## 3. Combination Scheme (Fixed Threshold)
Calibrates a single global threshold $T = \mu_{\text{absent}} + 0.30(\mu_{\text{present}} - \mu_{\text{absent}})$ to prevent ISI tail false alarms.

In [ ]:
res_combo_fixed = run_combination_fixed(
    bit_sequence=test_bits,
    num_molecule_types=5,
    delay_between_symbols=2.5,
    delay_between_molecules=0.4,
    plot=True,
)

## 4. Sparse Pattern Scheme (Molecule Exclusion)
Slots are partially filled with molecule types. Once a molecule wins a slot, it is excluded from later slots in the same symbol.

In [ ]:
res_sparse = run_sparse(
    bit_sequence=test_bits,
    num_molecule_types=5,
    delay_between_symbols=2.5,
    delay_between_molecules=0.4,
    plot=True,
)

## 5. Monte Carlo SER vs $T_s$ Sweep
Runs full pooled Monte Carlo simulations with 95% Wilson confidence intervals across all schemes and $N_{\text{Tx}}$ values.

In [ ]:
figures_dir = os.path.join(REPO_ROOT, "results", "figures")
os.makedirs(figures_dir, exist_ok=True)
out_fig = os.path.join(figures_dir, "ser_vs_ts_sweep.png")

results = run_ser_vs_ts_sweep(
    schemes=["permutation", "combo_fixed", "combo_adaptive", "sparse"],
    n_tx_list=[500, 1000],
    ts_values=[0.20, 0.25, 0.30, 0.35, 0.40],
    num_molecule_types=5,
    seq_length=100,
    num_trials=10,
    save_path=out_fig,
    log_scale=True,
)